# ds004752 V5.6 Tranche 2.1 Split Registry Lock and Feature Provenance

Purpose: lock subject-level split registry and populate source provenance after Tranche 2 scaffold artifacts pass review.

Integrity boundary: this notebook does not train models, does not materialize feature matrices, does not compute efficacy metrics, and does not open claims.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update repo into `/content`

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)

## 3. Confirm Tranche 2.1 code is present

In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -8

from pathlib import Path

required_files = [
    Path('src/v56/tranche2_lock.py'),
    Path('tests/unit/test_v56_tranche2_lock.py'),
    Path('bootstrap/run_v56_tranche2_lock.sh'),
    Path('docs/23_v56_tranche2_lock_runbook_2026-05-04.md'),
]
missing = [str(p) for p in required_files if not p.exists()]
print({'missing_required_files': missing})
assert not missing, missing

cli_text = Path('src/cli.py').read_text(encoding='utf-8')
assert 'v56-tranche2-lock' in cli_text
assert 'run_v56_tranche2_lock' in cli_text
print('V5.6 Tranche 2.1 split/provenance code is present.')

## 4. Configure source-of-truth paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260424T100159866284Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts'
SPLIT_REGISTRY_RUN = OUTPUT_ROOT / 'v56_split_registry/latest.txt'
FEATURE_PROVENANCE_RUN = OUTPUT_ROOT / 'v56_feature_provenance/latest.txt'

print('GATE0_RUN =', GATE0_RUN)
print('SPLIT_REGISTRY_RUN =', SPLIT_REGISTRY_RUN)
print('FEATURE_PROVENANCE_RUN =', FEATURE_PROVENANCE_RUN)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

for path in [GATE0_RUN, SPLIT_REGISTRY_RUN, FEATURE_PROVENANCE_RUN]:
    print(path, path.exists())
    assert path.exists(), f'Missing required path: {path}'

## 5. Preflight Gate 0 and Tranche 2 scaffold artifacts

In [ ]:
import json

manifest = json.loads((GATE0_RUN / 'manifest.json').read_text())
cohort_lock = json.loads((GATE0_RUN / 'cohort_lock.json').read_text())

split_scaffold_dir = Path(SPLIT_REGISTRY_RUN.read_text().strip())
feature_scaffold_dir = Path(FEATURE_PROVENANCE_RUN.read_text().strip())
split_scaffold = json.loads((split_scaffold_dir / 'v56_split_registry.json').read_text())
feature_scaffold = json.loads((feature_scaffold_dir / 'v56_feature_provenance.json').read_text())

preflight = {
    'gate0_manifest_status': manifest.get('manifest_status'),
    'gate0_blockers': manifest.get('gate0_blockers'),
    'cohort_lock_status': cohort_lock.get('cohort_lock_status'),
    'n_primary_eligible': cohort_lock.get('n_primary_eligible'),
    'split_scaffold_dir': str(split_scaffold_dir),
    'split_scaffold_status': split_scaffold.get('status'),
    'feature_scaffold_dir': str(feature_scaffold_dir),
    'feature_scaffold_status': feature_scaffold.get('status'),
}
print(json.dumps(preflight, indent=2))

assert preflight['gate0_manifest_status'] == 'signal_audit_ready', preflight
assert preflight['gate0_blockers'] == [], preflight
assert preflight['cohort_lock_status'] == 'signal_audit_ready', preflight
assert preflight['n_primary_eligible'] == 15, preflight
assert preflight['split_scaffold_status'] == 'pending_registry_lock', preflight
assert preflight['feature_scaffold_status'] == 'pending_feature_provenance_population', preflight
print('Preflight passed: Gate 0 and Tranche 2 scaffold artifacts are ready.')

## 6. Run V5.6 Tranche 2.1 split lock and feature provenance population

In [ ]:
%cd /content/eeg-ds004752
!bash bootstrap/run_v56_tranche2_lock.sh {GATE0_RUN} {SPLIT_REGISTRY_RUN} {FEATURE_PROVENANCE_RUN} {OUTPUT_ROOT}

## 7. Load latest Tranche 2.1 artifact dirs

In [ ]:
artifact_roots = {
    'v56_split_registry_lock': OUTPUT_ROOT / 'v56_split_registry_lock',
    'v56_feature_provenance_populated': OUTPUT_ROOT / 'v56_feature_provenance_populated',
}

artifact_dirs = {}
for family, root in artifact_roots.items():
    latest = root / 'latest.txt'
    assert latest.exists(), f'Missing latest pointer for {family}: {latest}'
    run_dir = Path(latest.read_text().strip())
    assert run_dir.exists(), f'Missing run dir for {family}: {run_dir}'
    artifact_dirs[family] = run_dir

for family, run_dir in artifact_dirs.items():
    print(f'{family}: {run_dir}')

## 8. Read artifact JSON and summaries

In [ ]:
split_lock_dir = artifact_dirs['v56_split_registry_lock']
feature_dir = artifact_dirs['v56_feature_provenance_populated']

split_lock = json.loads((split_lock_dir / 'v56_split_registry_lock.json').read_text())
split_summary = json.loads((split_lock_dir / 'v56_split_registry_lock_summary.json').read_text())
feature_provenance = json.loads((feature_dir / 'v56_feature_provenance_populated.json').read_text())
feature_summary = json.loads((feature_dir / 'v56_feature_provenance_populated_summary.json').read_text())

compact = {
    'split_lock': {
        'status': split_lock.get('status'),
        'claim_closed': split_lock.get('claim_closed'),
        'n_folds': len(split_lock.get('folds', [])),
        'n_primary_eligible': split_lock.get('n_primary_eligible'),
    },
    'feature_provenance': {
        'status': feature_provenance.get('status'),
        'claim_closed': feature_provenance.get('claim_closed'),
        'n_entries': len(feature_provenance.get('entries', [])),
        'missing_sources': feature_provenance.get('missing_sources'),
    },
}
print(json.dumps(compact, indent=2))

## 9. Integrity assertions

In [ ]:
assert split_lock['status'] == 'locked_subject_level_split_registry', split_lock
assert split_lock['claim_closed'] is True, split_lock
assert split_lock['subject_isolation_enforced'] is True, split_lock
assert split_lock['test_time_inference']['modality'] == 'scalp_eeg_only', split_lock
assert split_lock['test_time_inference']['allow_ieeg'] is False, split_lock
assert split_lock['test_time_inference']['allow_beamforming_bridge'] is False, split_lock
assert split_lock['n_primary_eligible'] == 15, split_lock
assert len(split_lock['folds']) == 30, len(split_lock['folds'])

for fold in split_lock['folds']:
    assert fold['outer_test_subject'] not in fold['train_subjects'], fold
    assert fold['test_time_allow_ieeg'] is False, fold
    assert fold['test_time_allow_beamforming_bridge'] is False, fold

assert feature_provenance['status'] == 'populated_source_hashes_and_split_links', feature_provenance
assert feature_provenance['claim_closed'] is True, feature_provenance
assert feature_provenance['required_links_satisfied']['split_registry'] is True, feature_provenance
assert feature_provenance['required_links_satisfied']['source_hashes'] is True, feature_provenance
assert feature_provenance['required_links_satisfied']['manifest'] is True, feature_provenance
assert feature_provenance['feature_matrix_materialized'] is False, feature_provenance
assert feature_provenance['model_training_run'] is False, feature_provenance
assert feature_provenance['efficacy_metrics_computed'] is False, feature_provenance
assert feature_provenance['claim_ready'] is False, feature_provenance

assert split_summary['claim_closed'] is True, split_summary
assert feature_summary['claim_closed'] is True, feature_summary
print('V5.6 Tranche 2.1 split/provenance artifacts passed integrity assertions.')

## 10. Print reports

In [ ]:
for path in [
    split_lock_dir / 'v56_split_registry_lock_report.md',
    feature_dir / 'v56_feature_provenance_populated_report.md',
]:
    print('')
    print('=' * 80)
    print(path)
    print('-' * 80)
    print(path.read_text()[:2500])

## 11. Decision gate closeout

In [ ]:
closeout = {
    'gate0_run': str(GATE0_RUN),
    'tranche2_1_status': 'split_registry_locked_and_feature_provenance_populated',
    'claim_closed': True,
    'feature_matrix_materialized': False,
    'model_training_run': False,
    'efficacy_metrics_computed': False,
    'split_registry_lock_dir': str(split_lock_dir),
    'feature_provenance_dir': str(feature_dir),
    'next_step': 'manual_review_then_plan_feature_matrix_or_baseline_execution_only_if_artifacts_pass',
    'not_allowed_next': [
        'do_not_train_RIFT_Net_Lite_yet',
        'do_not_run_A3_A4_comparators_yet',
        'do_not_open_efficacy_claim',
    ],
}
print(json.dumps(closeout, indent=2))